# Inpactor3-SegNT · Pipeline PanTEon cross-kingdom (Kaggle)

Entrena SegNT sobre PanTEon completo (plantas + hongos + animales) en Kaggle con GPU.

**Ventajas sobre Colab**:
- 30h/semana de GPU (vs 4h Colab)
- Sin límite de sesión
- Almacenamiento temporal ~20GB

**IMPORTANTE**: Los outputs se guardan en `/kaggle/working/` y puedes descargarlos directamente.

**Duración estimada**: ~6-8h GPU P100/T4

## 1. Verificar GPU disponible

In [ ]:
!nvidia-smi

## 2. Setup: clonar repo e instalar dependencias

In [ ]:
%cd /kaggle/working
!rm -rf Inpactor3-SegNT
!git clone https://github.com/Inpactor3/Inpactor3-SegNT.git
%cd Inpactor3-SegNT
!pip install -q -e .
!echo '✓ Instalación completada'

## 3. Descargar PanTEon database (~1 GB)

In [ ]:
import os
import subprocess

os.makedirs('data/raw', exist_ok=True)

print('Descargando PanTEon FASTA (v1.6.2)...')
!curl -Lk -o data/raw/PanTEon_Database_v1.6.2.fasta \
  "https://zenodo.org/api/records/21372179/files/PanTEon_Database_v1.6.2.fasta/content"

print('\nDescargando metadata...')
!curl -Lk -o data/raw/PanTEon_Database_metadata_v1.6.2.csv \
  "https://zenodo.org/api/records/21372179/files/PanTEon_Database_metadata_v1.6.2.csv/content"

!ls -lh data/raw/

## 4. Descargar genomas de fondo (~5 GB)

In [ ]:
!bash scripts/download_backgrounds.sh
!echo '\n✓ Genomas descargados'

## 5. Construir dataset común (~20-30 min)

In [ ]:
!python scripts/build_teon_dataset.py \
    --panteon data/raw/PanTEon_Database_v1.6.2.fasta \
    --metadata data/raw/PanTEon_Database_metadata_v1.6.2.csv \
    --genomes-dir data/common_dataset/genomes \
    --out-dir data/common_dataset \
    --n-positives 30000 \
    --n-negatives 30000

!ls -lh data/common_dataset/

## 6. Entrenar SegNT sobre PanTEon (~3-4 h en GPU)

In [ ]:
!python -m inpactor3_segnt.train_teon --config configs/nt_50m_teon.yaml
!echo '\n✓ Entrenamiento completado'

## 7. Descargar 4 genomas de validación (~4 GB, ~30 min)

In [ ]:
import os
os.makedirs('data/validation_genomes', exist_ok=True)

print('Descargando Arabidopsis (~120 MB)...')
!for CHR in 1 2 3 4 5; do \
  curl -Lk -o "data/validation_genomes/chr${CHR}.fa.gz" \
    "https://ftp.ensemblgenomes.org/pub/plants/release-58/fasta/arabidopsis_thaliana/dna/Arabidopsis_thaliana.TAIR10.dna_sm.chromosome.${CHR}.fa.gz"; \
  gunzip "data/validation_genomes/chr${CHR}.fa.gz"; \
done
!cat data/validation_genomes/chr*.fa > data/validation_genomes/Arabidopsis_thaliana.fasta
!rm data/validation_genomes/chr*.fa

print('\nDescargando Oryza sativa (~370 MB)...')
!curl -Lk -o data/validation_genomes/Oryza_sativa.fa.gz \
  "https://ftp.ensemblgenomes.org/pub/plants/release-58/fasta/oryza_sativa/dna/Oryza_sativa.IRGSP-1.0.dna_sm.toplevel.fa.gz"
!gunzip data/validation_genomes/Oryza_sativa.fa.gz

print('\nDescargando Ananas comosus (~500 MB)...')
!curl -Lk -o data/validation_genomes/Ananas_comosus.fa.gz \
  "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/001/540/865/GCF_001540865.1_ASM154086v1/GCF_001540865.1_ASM154086v1_genomic.fna.gz"
!gunzip data/validation_genomes/Ananas_comosus.fa.gz
!mv data/validation_genomes/GCF_001540865.1_ASM154086v1_genomic.fna data/validation_genomes/Ananas_comosus.fasta 2>/dev/null || echo 'Ya existe'

print('\nCopiando Zea mays...')
!cp data/common_dataset/genomes/Zea_mays.fasta data/validation_genomes/

!ls -lh data/validation_genomes/

## 8. Inferencia sobre los 4 genomas (~2 h)

In [ ]:
import os
os.makedirs('results/segnt', exist_ok=True)

genomes = ['Arabidopsis_thaliana', 'Oryza_sativa', 'Ananas_comosus', 'Zea_mays']

for genome in genomes:
    print(f'\n=== Prediciendo {genome} ===')
    !python -m inpactor3_segnt.predict \
        --checkpoint models/best_teon.pt \
        --genome data/validation_genomes/{genome}.fasta \
        --out results/segnt/{genome}_predictions.tab \
        --threshold 0.5

!ls -lh results/segnt/

## 9. Reporte de validación

In [ ]:
!python scripts/validate_on_genomes.py \
    --predictions-dir results/segnt \
    --genomes-dir data/validation_genomes \
    --model-name "Inpactor 3-SegNT (PanTEon cross-kingdom)" \
    --report results/validation_segnt.md

!cat results/validation_segnt.md

## 10. Copiar outputs a /kaggle/working para descargar

In [ ]:
import shutil
import os

# Copiar checkpoints y resultados al directorio raíz de working
print('Copiando archivos para descargar...')

shutil.copy('models/best_teon.pt', '/kaggle/working/best_teon.pt')
shutil.copytree('results', '/kaggle/working/results', dirs_exist_ok=True)
shutil.copytree('logs', '/kaggle/working/logs', dirs_exist_ok=True)

print('\n✓ Todos los outputs están listos en /kaggle/working/')
print('\nArchivos disponibles para descargar:')
!ls -lh /kaggle/working/ | grep -E '(best_teon|results|logs)'

## 11. Resumen de recursos utilizados

In [ ]:
!echo '=== Resumen de training ===' && \
echo 'Checkpoint: models/best_teon.pt' && \
du -sh models/best_teon.pt && \
echo '' && \
echo 'Resultados de predicción:' && \
du -sh results/segnt && \
echo '' && \
echo 'Logs de entrenamiento:' && \
du -sh logs